# SBER AI Journey — 3D Mesh Quality Control v2.1## End-to-End Training Pipeline on Google Colab**Competition Metric:**```f1_final = 10 x F1(quality) + 10 x F1_weighted(defects)```Maximum score: **20.0**### v2.1 Improvements| Technique | Description | Expected Gain ||-----------|-------------|:-------------:|| **EMA** | Exponential Moving Average of model weights (decay=0.999) | +0.3-0.8% F1 || **Multi-label Mixup** | Inter-sample blending adapted for multi-label (alpha=0.2) | +0.5-1.5% F1 || **Quality-Aware Thresholds** | Optimizes f1_final directly, not just F1_defects | Direct metric gain || **Temperature Scaling** | Post-training probability calibration via LBFGS | Better thresholds |**Expected runtime:** ~2-3.5 hours on T4 GPU**Expected F1_final:** ~17.5-18.0> **Important:** Change runtime to **GPU** (T4 or better) before running!

## 1. Setup & Install Dependencies

In [ ]:
%%capture!pip install -q torch torchvision numpy pandas scikit-learn scipy Pillow matplotlib tqdm gdown py7zr trimesh opencv-python-headless seaborn

In [ ]:
import sys, os, timeimport torchimport numpy as npimport pandas as pd# Verify GPUassert torch.cuda.is_available(), "GPU not available! Change runtime to GPU."print(f"PyTorch: {torch.__version__}")print(f"CUDA: {torch.version.cuda}")print(f"GPU: {torch.cuda.get_device_name(0)}")print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Add solution to pathSOLUTION_DIR = '/content/sber_mesh_qc/solution'os.makedirs(SOLUTION_DIR, exist_ok=True)sys.path.insert(0, SOLUTION_DIR)# Verify v2.1 features are availablefrom config import (    USE_EMA, EMA_DECAY, USE_MIXUP, MIXUP_ALPHA, MIXUP_PROB,    USE_TEMPERATURE_SCALING, TEMPERATURE_INIT,    OPTIMIZE_THRESHOLDS_F1_FINAL,    USE_EXTENDED_FEATURES, MESH_FEATURE_DIM_EXTENDED,    PROGRESSIVE_RESIZE, SEQUENTIAL_VIEW_PROCESSING,    USE_POINTNET_BRANCH,)print("=" * 50)print("  v2.1 Feature Flags")print("=" * 50)print(f"  EMA:                 {USE_EMA} (decay={EMA_DECAY})")print(f"  Mixup:               {USE_MIXUP} (alpha={MIXUP_ALPHA}, prob={MIXUP_PROB})")print(f"  Temperature Scaling: {USE_TEMPERATURE_SCALING} (init={TEMPERATURE_INIT})")print(f"  Quality-Aware Thresh:{OPTIMIZE_THRESHOLDS_F1_FINAL}")print(f"  Extended Features:   {USE_EXTENDED_FEATURES} ({MESH_FEATURE_DIM_EXTENDED}-dim)")print(f"  Progressive Resize:  {PROGRESSIVE_RESIZE}")print(f"  Sequential Views:    {SEQUENTIAL_VIEW_PROCESSING}")print(f"  PointNet Branch:     {USE_POINTNET_BRANCH}")print("=" * 50)

## 2. Download Competition Data

Downloads the train and test data from the direct SberCloud OBS archive (or Yandex.Disk as fallback).Contains: train.csv, test.csv, train/*.png, train/*.npz, test/*.png, test/*.npz

In [ ]:
import syssys.path.insert(0, SOLUTION_DIR)from data_utils import download_data, prepare_data_dirs, validate_data_integrityBASE_DIR = '/content/sber_mesh_qc'DATA_DIR = os.path.join(BASE_DIR, 'data')print("Downloading competition data (this may take 10-20 minutes)...")download_data(BASE_DIR)# Validatetrain_csv = os.path.join(DATA_DIR, 'train.csv')test_csv = os.path.join(DATA_DIR, 'test.csv')if os.path.isfile(train_csv) and os.path.isfile(test_csv):    train_dir = os.path.join(DATA_DIR, 'train')    test_dir = os.path.join(DATA_DIR, 'test')    if os.path.isdir(train_dir) and os.path.isdir(test_dir):        validate_data_integrity(train_csv, train_dir, test_csv, test_dir)# Quick statstrain_df = pd.read_csv(train_csv)test_df = pd.read_csv(test_csv)print(f"\nTrain: {len(train_df)} samples | Test: {len(test_df)} samples")print(f"Defect columns: {len(train_df.columns) - 2} (excluding item_id, quality)")print(f"Quality distribution: {train_df['quality'].value_counts().to_dict()}")

## 3. Extract 68-dim Geometric Mesh Features

Extracts 68 hand-crafted geometric features from .npz files (vertices + faces).Features include: bounding box, edges, face areas, volume, normals, topology,PCA shape analysis, spatial density, depth histograms, surface roughness.This step takes ~10-30 minutes but results are cached to .npz for reuse.

In [ ]:
import syssys.path.insert(0, SOLUTION_DIR)from mesh_features import batch_extract_mesh_featuresimport timesuffix = "extended" if USE_EXTENDED_FEATURES else "basic"feat_dim = MESH_FEATURE_DIM_EXTENDED if USE_EXTENDED_FEATURES else 58# ── Train features ──train_cache = os.path.join(BASE_DIR, f'mesh_features_train_{suffix}.npz')if os.path.exists(train_cache):    print(f"Loading cached train features from {train_cache}")    train_features = np.load(train_cache)['features']else:    train_ids = train_df['item_id'].tolist()    # Find NPZ directory    npz_dir = os.path.join(DATA_DIR, 'train')    if not any(f.endswith('.npz') for f in os.listdir(npz_dir)[:10]):        # Try subdirectory        for sub in ['npz', 'meshes']:            cand = os.path.join(npz_dir, sub)            if os.path.isdir(cand):                npz_dir = cand                break    print(f"Extracting {len(train_ids)} train features ({feat_dim}-dim)...")    t0 = time.time()    train_features = batch_extract_mesh_features(train_ids, npz_dir, extended=USE_EXTENDED_FEATURES)    print(f"Done in {time.time()-t0:.1f}s — shape: {train_features.shape}")    np.savez_compressed(train_cache, features=train_features)# ── Test features ──test_cache = os.path.join(BASE_DIR, f'mesh_features_test_{suffix}.npz')if os.path.exists(test_cache):    print(f"Loading cached test features from {test_cache}")    test_features = np.load(test_cache)['features']else:    test_ids = test_df['item_id'].tolist()    npz_dir = os.path.join(DATA_DIR, 'test')    if not any(f.endswith('.npz') for f in os.listdir(npz_dir)[:10]):        for sub in ['npz', 'meshes']:            cand = os.path.join(npz_dir, sub)            if os.path.isdir(cand):                npz_dir = cand                break    print(f"Extracting {len(test_ids)} test features...")    t0 = time.time()    test_features = batch_extract_mesh_features(test_ids, npz_dir, extended=USE_EXTENDED_FEATURES)    print(f"Done in {time.time()-t0:.1f}s — shape: {test_features.shape}")    np.savez_compressed(test_cache, features=test_features)print(f"\nTrain features: {train_features.shape}")print(f"Test features:  {test_features.shape}")

## 4. Train 5-Fold CV Ensemble

Trains a stratified cross-validation ensemble with all v2.1 features:- **EMA** for smoother, more generalizable weights- **Multi-label Mixup** for better generalization on imbalanced classes- **Progressive resize** (128px -> 192px -> 224px) for faster early epochs- **Focal Loss** with dynamic class weighting for severe imbalance- **Quality-aware threshold optimization** targeting f1_final directly- **Temperature scaling** for calibrated probabilitiesThis takes ~2-3.5 hours on T4.

In [ ]:
import syssys.path.insert(0, SOLUTION_DIR)from train import train_full_cvimport timeCHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints')LOG_DIR = os.path.join(BASE_DIR, 'logs')# Find train image directorytrain_image_dir = os.path.join(DATA_DIR, 'train')if not any(f.endswith('.png') for f in os.listdir(train_image_dir)[:5]):    for cand in [DATA_DIR, os.path.join(DATA_DIR, 'train_images')]:        if os.path.isdir(cand) and any(f.endswith('.png') for f in os.listdir(cand)[:5]):            train_image_dir = cand            breakprint(f"Training configuration:")print(f"  Samples: {len(train_df)}")print(f"  Image dir: {train_image_dir}")print(f"  Checkpoint dir: {CHECKPOINT_DIR}")print(f"  Log dir: {LOG_DIR}")print(f"  Mesh features: {train_features.shape}")print(f"\nStarting CV training...")t0 = time.time()cv_results = train_full_cv(    train_df=train_df,    image_dir=train_image_dir,    mesh_features=train_features,    checkpoint_dir=CHECKPOINT_DIR,    log_dir=LOG_DIR,)elapsed = time.time() - t0print(f"\nTraining completed in {elapsed/60:.1f} minutes")print(f"\nCV Results Summary:")avg = cv_results['avg_metrics']print(f"  F1_final:   {avg['f1_final_mean']:.2f} +/- {avg['f1_final_std']:.2f}")print(f"  F1_quality: {avg['f1_quality_mean']:.4f} +/- {avg['f1_quality_std']:.4f}")print(f"  F1_defects: {avg['f1_defects_mean']:.4f} +/- {avg['f1_defects_std']:.4f}")print(f"  Temperature: {cv_results['avg_temperature']:.3f}")

## 5. Generate Submission

Runs ensemble inference with:- CV model averaging (using EMA weights)- Test-Time Augmentation (2 flips x 2 rotations = 4 variants)- Temperature-scaled probabilities (learned from CV)- Quality-aware optimized thresholds

In [ ]:
import syssys.path.insert(0, SOLUTION_DIR)from inference import generate_submissionSUBMISSION_PATH = os.path.join(BASE_DIR, 'submission.csv')CV_RESULTS_PATH = os.path.join(LOG_DIR, 'cv_results.json')# Find test image directorytest_image_dir = os.path.join(DATA_DIR, 'test')if not any(f.endswith('.png') for f in os.listdir(test_image_dir)[:5]):    for cand in [DATA_DIR, os.path.join(DATA_DIR, 'test_images')]:        if os.path.isdir(cand) and any(f.endswith('.png') for f in os.listdir(cand)[:5]):            test_image_dir = cand            breakgenerate_submission(    test_csv_path=os.path.join(DATA_DIR, 'test.csv'),    test_image_dir=test_image_dir,    checkpoint_dir=CHECKPOINT_DIR,    output_path=SUBMISSION_PATH,    cv_results_path=CV_RESULTS_PATH,    mesh_features=test_features,)

## 6. Download Submission

In [ ]:
import pandas as pdfrom google.colab import filessubmission = pd.read_csv(SUBMISSION_PATH)print(f"Submission shape: {submission.shape}")print(f"Columns: {list(submission.columns)}")print(f"\nFirst 5 rows:")print(submission.head())print(f"\nPrediction statistics:")print(f"  Good (quality=1): {submission['quality'].sum()} ({submission['quality'].mean()*100:.1f}%)")print(f"  Bad (quality=0): {(1-submission['quality']).sum()} ({(1-submission['quality'].mean())*100:.1f}%)")# Downloadfiles.download(SUBMISSION_PATH)print("\nsubmission.csv downloaded!")

In [ ]:
# Also download the probability file for analysisproba_path = SUBMISSION_PATH.replace('.csv', '_proba.csv')if os.path.exists(proba_path):    files.download(proba_path)    print("submission_proba.csv downloaded!")